In [16]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import joblib

In [3]:
train_df = pd.read_csv('../Data/train.csv')
test_df = pd.read_csv('../Data/test.csv')
validation_df = pd.read_csv('../Data/validation.csv')

# Separate target column
train_X = train_df.drop(columns=['price'])
train_Y = train_df['price']

test_X = test_df.drop(columns=['price'])
test_Y = test_df['price']

validation_X = validation_df.drop(columns=['price'])
validation_Y = validation_df['price']

Se va a realizar un experimento donde se varian hiperparametros y se busca la mejor solucion

In [4]:
# Create a dictionary to store the results
results = []

# Define the hyperparameter values to test
n_estimators_values = [50, 100]
max_depth_values = [5, 10]
min_samples_split_values = [2, 5]

Se entrena el modelo con todos los posibles valores de hiperparametros y se selecciona el que tenga mejores métricas

In [7]:
# Loop through combinations of hyperparameters
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        for min_samples_split in min_samples_split_values:
            # Initialize the model with the hyperparameters
            model = RandomForestRegressor(n_estimators=n_estimators,
                                          max_depth=max_depth,
                                          min_samples_split=min_samples_split,
                                          random_state=42)

            # Train the model
            model.fit(train_X, train_Y)

            # Predict on the validation set
            validation_pred = model.predict(validation_X)

            # Calculate the metrics
            mse = mean_squared_error(validation_Y, validation_pred)
            r2 = r2_score(validation_Y, validation_pred)

            # Save the results
            results.append({
                'n_estimators': n_estimators,
                'max_depth': max_depth,
                'min_samples_split': min_samples_split,
                'MSE': mse,
                'R2': r2
            })

In [9]:
# Convert the results to a DataFrame
results_df = pd.DataFrame(results)
results_df

,n_estimators,max_depth,min_samples_split,MSE,R2
0,50,5,2,2.500556e+10,0.761711
1,50,5,5,2.500364e+10,0.761729
2,50,10,2,1.699409e+10,0.838056
3,50,10,5,1.687691e+10,0.839172
4,100,5,2,2.510939e+10,0.760721
5,100,5,5,2.510710e+10,0.760743
6,100,10,2,1.705381e+10,0.837486
7,100,10,5,1.700591e+10,0.837943


In [12]:
# Select the best hyperparameter combination based on the R2 metric
best_params = results_df.loc[results_df['R2'].idxmax()]
best_params

n_estimators         5.000000e+01
max_depth            1.000000e+01
min_samples_split    5.000000e+00
MSE                  1.687691e+10
R2                   8.391723e-01
Name: 3, dtype: float64

Se entrena el modelo con los mejores parámetros.

In [ ]:
# Train the model with the best hyperparameters on the validation set
best_model = RandomForestRegressor(n_estimators=int(best_params['n_estimators']),
                                   max_depth=int(best_params['max_depth']) if best_params['max_depth'] is not None else None,
                                   min_samples_split=int(best_params['min_samples_split']),
                                   random_state=42)

best_model.fit(validation_X, validation_Y)

## Finalmente se usan los datos de "test" para validar las métricas del modelo.

In [14]:
# Evaluate the model on the test set
test_pred = best_model.predict(test_X)
test_mse = mean_squared_error(test_Y, test_pred)
test_r2 = r2_score(test_Y, test_pred)

# Display the final model metrics on the test set
print(f"Test MSE: {test_mse}")
print(f"Test R2: {test_r2}")

Test MSE: 20645129139.85943
Test R2: 0.8023211583569384


Como ejemplo: se tiene un nuevo inmueble dado por los siguientes datos y se debe predecir el precio.

In [15]:
new_data = {
    'bedrooms': [3],
    'bathrooms': [2],
    'sqft_living': [1500],
    'sqft_lot': [5000],
    'floors': [2],
    'condition': [3],
    'grade': [7],
    'sqft_above': [1200],
    'yr_built': [2005],
    'zipcode': [98003],
    'lat': [47.5],
    'long': [-122.3],
    'sqft_living15': [1500],
    'sqft_lot15': [5000],
    'year': [2025],
    'month': [4]
}

# Convertir el nuevo dato en un DataFrame
new_df = pd.DataFrame(new_data)

# Usar el modelo entrenado (best_model) para hacer la predicción
predicted_price = best_model.predict(new_df)

# Mostrar el resultado de la predicción
print(f"Predicted Price: {predicted_price[0]}")

Predicted Price: 282405.83220838755


Para guardar el modelo y que se pueda usar despues se pueden usar herramientas como joblib

In [17]:
joblib.dump(best_model, 'random_forest_model.pkl')
print("Model saved successfully!")

Model saved successfully!


Para usarlo:

In [18]:
# Cargar el modelo guardado
loaded_model = joblib.load('random_forest_model.pkl')

# Ahora puedes usar el modelo cargado para hacer predicciones
predicted_price = loaded_model.predict(new_df)
print(f"Predicted Price: {predicted_price[0]}")

Predicted Price: 282405.83220838755
